In [2]:
import pandas as pd

weather = pd.read_csv("../data/processed/smhi_weather_clean.csv", parse_dates=["datetime"])
daily_mean = weather.groupby(weather["datetime"].dt.date)["temperature_c"].mean()
daily_mean.index = pd.to_datetime(daily_mean.index)

candidates = daily_mean[
    (daily_mean.index >= "2021-01-01") &
    (daily_mean.index <= "2026-08-31")
]

bins = pd.cut(candidates, bins=range(-20, 31, 5))
print(bins.value_counts().sort_index())
print(f"\nTotal candidate dates (full year, no month filter): {len(candidates)}")

temperature_c
(-20, -15]      0
(-15, -10]      9
(-10, -5]      68
(-5, 0]       247
(0, 5]        433
(5, 10]       396
(10, 15]      338
(15, 20]      364
(20, 25]      120
(25, 30]        3
Name: count, dtype: int64

Total candidate dates (full year, no month filter): 1978


In [4]:
import sys
sys.path.insert(0, "../src")

from select_dates import select_dates

In [7]:
import numpy as np

YEAR_ROUND_BANDS = {
    "(-15,-10]": (-15, -10, 9),
    "(-10,-5]": (-10, -5, 15),
    "(-5,0]": (-5, 0, 15),
    "(0,5]": (0, 5, 15),
    "(5,10]": (5, 10, 15),
    "(10,15]": (10, 15, 15),
    "(15,20]": (15, 20, 15),
    "(20,25]": (20, 25, 15),
    "(25,30]": (25, 30, 3),
}

backfill_dates = select_dates(
    weather_path="../data/processed/smhi_weather_clean.csv",
    bands=YEAR_ROUND_BANDS,
    month_filter=None,
)
print(f"Total selected: {len(backfill_dates)} dates")

Total selected: 114 dates


In [8]:
dt = pd.to_datetime(backfill_dates)
temps_check = pd.Series(dt).apply(lambda d: daily_mean.loc[d])
print(pd.cut(temps_check, bins=range(-20, 31, 5)).value_counts().sort_index())

(-20, -15]     0
(-15, -10]     6
(-10, -5]     15
(-5, 0]       15
(0, 5]        15
(5, 10]       15
(10, 15]      15
(15, 20]      15
(20, 25]      15
(25, 30]       3
Name: count, dtype: int64


In [9]:
with open("../reports/backfill_dates.txt", "w") as f:
    f.write("\n".join(backfill_dates))
print(f"Saved {len(backfill_dates)} dates to reports/backfill_dates.txt")

Saved 114 dates to reports/backfill_dates.txt
